# 03 - The BRCA example

Same as `BRCA_example/tutorialScript.m`:

```matlab
consistentRxnsBool = fastcc(model, 1e-4, 1);
consistentModel    = removeRxns(model, model.rxns(setdiff(1:numel(model.rxns), consistentRxnsBool)));
consistentModel    = changeObjective(consistentModel, "biomass_reaction");
discretized        = discretizeFPKM(fpkm, colnames);
[BCRAmodel, retainedRxns] = rFastcormics(consistentModel, discretized, rownames, dico, ...
                                          'biomass_reaction', 0.9, 1e-4, optionalSettings);
```

Solver: the MATLAB reference was produced with Gurobi, so the default backend here is Gurobi
too, with COBRA's defaults (`FeasibilityTol = OptimalityTol = 1e-6`, `Method = -1`).
Every FASTCORE step picks one vertex out of many optimal ones, so a different solver gives a
slightly different reaction set. `rf.set_default_solver("highs")` works without a licence but
will drift by ~1%.

In [2]:
import json, pickle, time
import numpy as np
import pandas as pd
import rfastcormics_v2 as rf

rf.set_default_solver("gurobi")      # "highs" if no Gurobi licence

def load_model(path="data/model.pkl"):
    m = pickle.load(open(path, "rb"))
    return rf.CobraModel(
        S=m["S"], rxns=m["rxns"], mets=m["mets"], lb=m["lb"], ub=m["ub"], c=m["c"],
        b=m["b"], csense=m["csense"], osense=m["osense"], genes=m["genes"],
        rules=m["rules"], grRules=m["grRules"], rxnGeneMat=m["rxnGeneMat"],
        subSystems=m["subSystems"], metFormulas=m["metFormulas"],
        metNames=m["metNames"], rxnNames=m["rxnNames"])

model = load_model()
exp = np.load("data/expression.npz", allow_pickle=True)
ref = np.load("data/matlab_reference.npz", allow_pickle=True)
dico = pd.read_csv("data/dico.csv", dtype=str)
settings = json.load(open("data/optional_settings.json"))
model

<CobraModel 5837 mets x 10601 rxns, 2248 genes>

## Consistency check

In [3]:
t = time.time()
consistent_rxns, _, _ = rf.fastcc(model, epsilon=1e-4, print_level=1)
print("fastcc %.1f s" % (time.time() - t))

matlab_consistent = np.sort(ref["consistentRxnsBool"] - 1)     # MATLAB is 1-based
print("python:", consistent_rxns.size, " matlab:", matlab_consistent.size,
      " identical:", np.array_equal(consistent_rxns, matlab_consistent))
print("dropped:", model.rxns[np.setdiff1d(np.arange(model.n_rxns), consistent_rxns)])

 10601	Total reactions
  5239	Reversible reactions.
  5362	Irreversible reactions.
fastcc 3.0 s
python: 10600  matlab: 10600  identical: True
dropped: ['r12989']


In [4]:
drop = np.setdiff1d(np.arange(model.n_rxns), consistent_rxns)
consistent_model, _ = rf.remove_rxns(model, model.rxns[drop])
consistent_model = rf.change_objective(consistent_model, "biomass_reaction")
consistent_model

<CobraModel 5835 mets x 10600 rxns, 2248 genes>

`changeObjective` is not optional here: `findExcRxns.m` excludes the objective reaction
from the exchange set, so getting it wrong changes the medium constraints.

## Discretisation

In [5]:
discretized = rf.discretize_fpkm(exp["fpkm"], exp["colnames"], verbose=False)[0]
# or, to take the discretisation out of the picture entirely:
# discretized = exp["discretized_matlab"]
rownames = exp["rownames"]
dict(zip(*np.unique(discretized, return_counts=True)))

{np.float64(-1.0): np.int64(183715),
 np.float64(0.0): np.int64(236870),
 np.float64(1.0): np.int64(163615)}

## rFastcormics

In [6]:
opt = settings["optionalSettings"]
t = time.time()
BCRAmodel, retained_rxns, core_rxns = rf.r_fastcormics(
    consistent_model, discretized, rownames, dico,
    biomass_reaction_name="biomass_reaction",
    consensus_proportion=0.9, epsilon=1e-4, optional_settings=opt)
print("rFastcormics %.1f s" % (time.time() - t))
BCRAmodel

Number of uptake reactions associated with medium metabolites: 50.
Warning metabolites with X in their Formulas these inputs are closed
Warning metabolites with Y in their Formulas these inputs are closed
Transcripts will temporarily be removed to allow matching with the discretized data.
2119 of 2248 genes of the model matched
Number of core reactions (after mapping, transporters removed, without .func reactions): 1446
Number of non expressed reactions (after mapping): 509.
Model still contains biomass_reaction after application of medium constraints, and FBA result is not null.
Medium is sufficient. Continuing with fastcore.
Model additional uptakes (not provided by the medium):
EX_hdl_hs[e]
EX_nh4[e]
EX_o2s[e]
EX_ppi[e]
rFastcormics is done.
rFastcormics 472.0 s


<CobraModel 2133 mets x 2578 rxns, 1283 genes>

## Compare with MATLAB

In [7]:
def compare(py_idx, matlab_key, model_ref, name):
    ml = np.sort(ref[matlab_key] - 1)
    py = np.sort(np.asarray(py_idx))
    inter = np.intersect1d(py, ml); union = np.union1d(py, ml)
    print(f"--- {name} ---")
    print(f"python {py.size} rxns / matlab {ml.size} rxns")
    print(f"identical: {np.array_equal(py, ml)}   intersection {inter.size}   "
          f"jaccard {inter.size/union.size:.4f}")
    only_py = model_ref.rxns[np.setdiff1d(py, ml)]
    only_ml = model_ref.rxns[np.setdiff1d(ml, py)]
    if only_py.size: print("python only:", list(only_py[:30]))
    if only_ml.size: print("matlab only:", list(only_ml[:30]))
    return py, ml

py_main, ml_main = compare(retained_rxns, "retainedRxns", consistent_model, "main")

--- main ---
python 2578 rxns / matlab 2584 rxns
identical: False   intersection 2549   jaccard 0.9755
python only: ['3DPHBH1', '3MOBt2im', 'ACGALK', 'CHSTEROLt1', 'CITtam', 'DNDPt21m', 'GLYCK2', 'MMCDm', 'NDPK5n', 'PHCDm', 'S4T5g', 'S6T24g', 'r0504', 'r0525', 'r1561', 'r1563', 'r1571', 'r1993', 'r2009', 'r2313', 'r2435', 'PUNP2', 'UGLT', 'HMR_0839', 'HMR_0841', 'HMR_0854', 'HMR_2673', 'HMR_2715', 'sink_met_L[c]']
matlab only: ['3DPHBH2', '3MOPt2im', 'ACGALK2', 'ARACHDt2', 'C160CRNt', 'C4STMO2Pr', 'CHSTEROLt2', 'DADA', 'DNDPt42m', 'ILEtec', 'LNELDCCRNt', 'LYStiDF', 'MMEm', 'S4T3g', 'S6T23g', 'SACCD4m', 'STRDNCCRNt', 'TYRt', 'r0399', 'r0502', 'r0686', 'r0801', 'r0913', 'r1599', 'TRPt', 'ARGtD', 'EX_met_L[e]', 'PUNP6', 'r0339', 'GALT']


In [8]:
py_names = set(map(str, BCRAmodel.rxns))
ml_names = set(map(str, ref["BCRAmodel_rxns"]))
print("python:", len(py_names), " matlab:", len(ml_names),
      " same:", py_names == ml_names, " intersection:", len(py_names & ml_names))

python: 2578  matlab: 2584  same: False  intersection: 2549


## The other two cases from the tutorial

In [9]:
# insufficient medium -> triggers the medium-filling second fastcore run
optbis = settings["optionalSettingsBis"]
BCRAmodelBis, retained_bis, _ = rf.r_fastcormics(
    consistent_model, discretized, rownames, dico, "biomass_reaction", 0.9, 1e-4, optbis)
compare(retained_bis, "retainedRxnsBis", consistent_model, "insufficient medium");

Number of uptake reactions associated with medium metabolites: 32.
Warning metabolites with X in their Formulas these inputs are closed
Warning metabolites with Y in their Formulas these inputs are closed
Transcripts will temporarily be removed to allow matching with the discretized data.
2119 of 2248 genes of the model matched
Number of core reactions (after mapping, transporters removed, without .func reactions): 1446
Number of non expressed reactions (after mapping): 509.


In [ ]:
# no optionalSettings at all
BCRAmodelTer, retained_ter, _ = rf.r_fastcormics(
    consistent_model, discretized, rownames, dico, "biomass_reaction", 0.9, 1e-4)
compare(retained_ter, "retainedRxnsTer", consistent_model, "no optionalSettings");

## Export

Convert to cobrapy for FBA / deletions / SBML. Notebook 04 picks up from the exported file.

In [ ]:
import cobra
from cobra.flux_analysis import single_gene_deletion

cb_ctx = rf.to_cobrapy(BCRAmodel, "BRCA_context_specific")
cb_ctl = rf.to_cobrapy(consistent_model, "consistent_control")
for m in (cb_ctx, cb_ctl):
    m.solver = "gurobi"

print("context-specific:", len(cb_ctx.reactions), "rxns,", len(cb_ctx.metabolites),
      "mets,", len(cb_ctx.genes), "genes")
print("FBA control          :", cb_ctl.optimize().objective_value)
print("FBA context-specific :", cb_ctx.optimize().objective_value)

Set parameter Username
Set parameter LicenseID to value 2806479
Academic license - for non-commercial use only - expires 2027-04-13


In [ ]:
cobra.io.write_sbml_model(cb_ctx, "BRCA_context_specific_model.xml")
cobra.io.save_json_model(cb_ctx, "BRCA_context_specific_model.json")

## Notes

- `fastcc` matches MATLAB exactly on this model (10600 of 10601, same reaction dropped).
- The reaction x sample mapping matrix built from the Python discretisation is identical to the
  one built from MATLAB's, so the handful of flipped discretisation cells never propagate.
- Core after removing transporters and adding `.func` is 1449 reactions; 509 reactions get
  knocked out as non-expressed. Both match the MATLAB log.
- `removeRxns` + `removeUnusedGenes` reproduce `BCRAmodel`'s dimensions exactly
  (2584 / 2134 / 1290) when handed MATLAB's own reaction list.
- The final reaction set is the only thing that drifts, and only because LP7/LP10 are
  degenerate - unique optimal value, many optimal vertices. Gurobi lands around 2578 vs
  MATLAB's 2584 (jaccard 0.976); HiGHS around 2575 (0.970). The differences come in matched
  pairs (`3DPHBH1`/`3DPHBH2`, `ACGALK`/`ACGALK2`, `CHSTEROLt1`/`CHSTEROLt2`), i.e. equivalent
  alternatives, not a logic difference. Notebook 04 checks whether the models are equivalent
  where it actually matters (growth rate, essential genes).